In [1]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm

DATASET_PATH = Path(r"D:\FINAL DFDT\SELECTED_DATASET")
SPLIT_PATH = DATASET_PATH / "splits"

split_manifest = pd.read_csv(
    SPLIT_PATH / "dataset_split_manifest.csv"
)

print("Total images:", len(split_manifest))
print("Split distribution:")
print(split_manifest["split"].value_counts())

print("\nDescriptor extraction environment ready.")

Total images: 10000
Split distribution:
split
train         7000
test          1500
validation    1500
Name: count, dtype: int64

Descriptor extraction environment ready.


In [ ]:
import numpy as np
import cv2

def extract_degradation_descriptors(image_path):

    # Load image as RGB
    image = cv2.imread(str(image_path))

    if image is None:
        raise ValueError(f"Cannot read image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    gray_float = gray.astype(np.float32) / 255.0

    # 1. Brightness
    brightness = np.mean(gray_float)

    # 2. Contrast
    contrast = np.std(gray_float)

    # 3. Sharpness (Laplacian variance)
    sharpness = cv2.Laplacian(
        gray_float, cv2.CV_32F
    ).var()

    # 4. Residual energy
    blurred = cv2.GaussianBlur(
        gray_float, (5, 5), 0
    )

    residual = gray_float - blurred

    residual_energy = np.mean(residual ** 2)

    # 5. Absolute residual
    absolute_residual = np.mean(np.abs(residual))

    # 6–8. FFT frequency energy
    fft = np.fft.fftshift(
        np.fft.fft2(gray_float)
    )

    magnitude = np.abs(fft) ** 2

    height, width = gray.shape

    center_y = height // 2
    center_x = width // 2

    y, x = np.ogrid[:height, :width]

    distance = np.sqrt(
        (y - center_y) ** 2 +
        (x - center_x) ** 2
    )

    max_distance = np.sqrt(
        center_y ** 2 + center_x ** 2
    )

    normalized_distance = distance / max_distance

    low_mask = normalized_distance < 0.15

    mid_mask = (
        (normalized_distance >= 0.15) &
        (normalized_distance < 0.50)
    )

    high_mask = normalized_distance >= 0.50

    total_energy = np.sum(magnitude) + 1e-8

    low_frequency_energy = np.sum(
        magnitude[low_mask]
    ) / total_energy

    mid_frequency_energy = np.sum(
        magnitude[mid_mask]
    ) / total_energy

    high_frequency_energy = np.sum(
        magnitude[high_mask]
    ) / total_energy

    return np.array([
        brightness,
        contrast,
        sharpness,
        residual_energy,
        absolute_residual,
        low_frequency_energy,
        mid_frequency_energy,
        high_frequency_energy
    ], dtype=np.float32)

In [6]:
from pathlib import Path

DATASET_PATH = Path(r"D:\FINAL DFDT\SELECTED_DATASET")

split_manifest = pd.read_csv(
    DATASET_PATH / "splits" / "dataset_split_manifest.csv"
)

# Reconstruct image paths
split_manifest["image_path"] = split_manifest.apply(
    lambda row: str(
        DATASET_PATH /
        row["class"].lower() /
        Path(row["destination_image"]).name
    ),
    axis=1
)

print("Columns:", split_manifest.columns.tolist())
print("Total images:", len(split_manifest))
print("Image path column created:", "image_path" in split_manifest.columns)

print("Missing images:", sum(
    not Path(p).exists()
    for p in split_manifest["image_path"]
))

Columns: ['class', 'source_subfolder', 'source_image', 'destination_image', 'split', 'image_path']
Total images: 10000
Image path column created: True
Missing images: 0


In [7]:
sample_images = split_manifest.sample(
    n=6, random_state=42
).reset_index(drop=True)

descriptor_names = [
    "Brightness",
    "Contrast",
    "Sharpness",
    "Residual Energy",
    "Absolute Residual",
    "FFT Low",
    "FFT Mid",
    "FFT High"
]

sample_results = []

for _, row in sample_images.iterrows():

    features = extract_degradation_descriptors(
        row["image_path"]
    )

    result = {
        "Class": row["class"],
        "Split": row["split"]
    }

    result.update(dict(zip(descriptor_names, features)))

    sample_results.append(result)

sample_df = pd.DataFrame(sample_results)

display(sample_df.round(6))

,Class,Split,Brightness,Contrast,Sharpness,Residual Energy,Absolute Residual,FFT Low,FFT Mid,FFT High
0,FAKE,validation,0.484520,0.183493,0.001118,0.000160,0.005548,0.996259,0.003453,0.000289
1,REAL,train,0.575782,0.101227,0.000790,0.000104,0.004438,0.998590,0.001365,0.000044
2,REAL,train,0.415602,0.155017,0.001165,0.000183,0.007053,0.994209,0.005518,0.000274
3,REAL,train,0.429977,0.273540,0.005501,0.000623,0.012750,0.989424,0.009725,0.000850
4,REAL,train,0.267470,0.315937,0.001650,0.000199,0.005777,0.993077,0.006338,0.000585
5,FAKE,train,0.424602,0.206524,0.007845,0.000629,0.009600,0.991474,0.007484,0.001042


In [8]:
feature_values = sample_df[descriptor_names].to_numpy()

print("Feature matrix shape:", feature_values.shape)
print("NaN values:", np.isnan(feature_values).sum())
print("Infinite values:", np.isinf(feature_values).sum())

print("\nFFT energy sum:")

fft_sum = (
    sample_df["FFT Low"] +
    sample_df["FFT Mid"] +
    sample_df["FFT High"]
)

print(fft_sum)

assert feature_values.shape == (6, 8)
assert np.isfinite(feature_values).all()
assert np.allclose(fft_sum, 1.0, atol=1e-5)

print("\nALL 8 DESCRIPTORS VALIDATED!")

Feature matrix shape: (6, 8)
NaN values: 0
Infinite values: 0

FFT energy sum:
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
5    1.0
dtype: float32

ALL 8 DESCRIPTORS VALIDATED!


In [9]:
from tqdm import tqdm

all_features = []

for _, row in tqdm(
    split_manifest.iterrows(),
    total=len(split_manifest),
    desc="Extracting descriptors"
):

    features = extract_degradation_descriptors(
        row["image_path"]
    )

    record = {
        "filename": Path(row["image_path"]).name,
        "class": row["class"],
        "label": row["label"] if "label" in row else (
            0 if row["class"] == "REAL" else 1
        ),
        "split": row["split"],
        "source_subfolder": row["source_subfolder"]
    }

    for name, value in zip(descriptor_names, features):
        record[name] = float(value)

    all_features.append(record)

features_df = pd.DataFrame(all_features)

print("Extraction completed!")
print("Feature table shape:", features_df.shape)

display(features_df.head())

Extracting descriptors: 100%|██████████| 10000/10000 [00:51<00:00, 192.58it/s]

Extraction completed!
Feature table shape: (10000, 13)


,filename,class,label,split,source_subfolder,Brightness,Contrast,Sharpness,Residual Energy,Absolute Residual,FFT Low,FFT Mid,FFT High
0,000_01_191.png,REAL,0,train,000,0.639689,0.238433,0.001665,0.000246,0.007512,0.996794,0.003058,0.000148
1,000_02_178.png,REAL,0,train,000,0.661405,0.241862,0.002159,0.000317,0.008389,0.996276,0.003534,0.000190
2,000_03_101.png,REAL,0,train,000,0.646459,0.242665,0.002124,0.000317,0.008547,0.995577,0.004174,0.000249
3,000_04_076.png,REAL,0,train,000,0.653538,0.244480,0.001755,0.000265,0.007934,0.996680,0.003190,0.000130
4,000_05_063.png,REAL,0,train,000,0.678674,0.243192,0.001894,0.000285,0.008266,0.996832,0.003056,0.000112


In [10]:
print("DESCRIPTOR STATISTICS")

display(
    features_df[descriptor_names].describe().T.round(6)
)

print("\nNaN values:")
print(features_df[descriptor_names].isna().sum())

print("\nInfinite values:")
print(
    np.isinf(
        features_df[descriptor_names].to_numpy()
    ).sum()
)

DESCRIPTOR STATISTICS


,count,mean,std,min,25%,50%,75%,max
Brightness,10000.0,0.410900,0.112854,0.130840,0.329429,0.398058,0.477290,0.834523
Contrast,10000.0,0.203546,0.046830,0.075719,0.170265,0.198886,0.230702,0.381184
Sharpness,10000.0,0.002203,0.003026,0.000052,0.000613,0.001182,0.002491,0.040996
Residual Energy,10000.0,0.000255,0.000294,0.000005,0.000092,0.000165,0.000306,0.004136
Absolute Residual,10000.0,0.006732,0.003052,0.001439,0.004793,0.006060,0.007870,0.037807
FFT Low,10000.0,0.993661,0.005169,0.903980,0.992329,0.994720,0.996519,0.999612
FFT Mid,10000.0,0.005860,0.004821,0.000372,0.003237,0.004852,0.007076,0.088272
FFT High,10000.0,0.000478,0.000478,0.000016,0.000213,0.000354,0.000597,0.009410



NaN values:
Brightness           0
Contrast             0
Sharpness            0
Residual Energy      0
Absolute Residual    0
FFT Low              0
FFT Mid              0
FFT High             0
dtype: int64

Infinite values:
0


In [11]:
OUTPUT_PATH = DATASET_PATH / "degradation_features"
OUTPUT_PATH.mkdir(exist_ok=True)

features_df.to_csv(
    OUTPUT_PATH / "degradation_features_10000.csv",
    index=False
)

print("Feature dataset saved:")
print(OUTPUT_PATH / "degradation_features_10000.csv")

Feature dataset saved:
D:\FINAL DFDT\SELECTED_DATASET\degradation_features\degradation_features_10000.csv


In [12]:

# STEP 8: NORMALIZE DESCRIPTORS USING TRAINING DATA ONLY

descriptor_names = [
    "Brightness",
    "Contrast",
    "Sharpness",
    "Residual Energy",
    "Absolute Residual",
    "FFT Low",
    "FFT Mid",
    "FFT High"
]

# Separate training data
train_mask = features_df["split"].str.lower() == "train"

train_features = features_df.loc[train_mask, descriptor_names]

# Calculate statistics using ONLY training data
descriptor_mean = train_features.mean()
descriptor_std = train_features.std()

# Prevent division by zero
descriptor_std = descriptor_std.replace(0, 1.0)

# Apply training statistics to all splits
features_normalized = features_df.copy()

features_normalized[descriptor_names] = (
    features_df[descriptor_names] - descriptor_mean
) / descriptor_std

# Verify results
print("Training samples:", len(train_features))
print("\nNormalized training descriptor statistics:")

display(
    features_normalized.loc[train_mask, descriptor_names]
    .describe()
    .T
    .round(4)
)

print(
    "\nNaN values:",
    features_normalized[descriptor_names].isna().sum().sum()
)

print(
    "Infinite values:",
    np.isinf(
        features_normalized[descriptor_names].to_numpy()
    ).sum()
)


Training samples: 7000

Normalized training descriptor statistics:


,count,mean,std,min,25%,50%,75%,max
Brightness,7000.0,-0.0,1.0,-2.4426,-0.7078,-0.1122,0.5657,3.7309
Contrast,7000.0,0.0,1.0,-2.5542,-0.7217,-0.0834,0.5831,3.8545
Sharpness,7000.0,-0.0,1.0,-0.7324,-0.5468,-0.3371,0.1171,10.1597
Residual Energy,7000.0,-0.0,1.0,-0.8858,-0.5814,-0.3037,0.2086,9.6506
Absolute Residual,7000.0,-0.0,1.0,-1.6600,-0.6652,-0.2069,0.4287,7.1164
FFT Low,7000.0,-0.0,1.0,-16.9901,-0.2479,0.1862,0.5381,1.1203
FFT Mid,7000.0,0.0,1.0,-1.1123,-0.5309,-0.1936,0.2488,16.8348
FFT High,7000.0,0.0,1.0,-0.9189,-0.5332,-0.2534,0.2177,17.5119



NaN values: 0
Infinite values: 0


In [13]:

import json

OUTPUT_PATH = DATASET_PATH / "degradation_features"
OUTPUT_PATH.mkdir(exist_ok=True)

# Save normalized features
features_normalized.to_csv(
    OUTPUT_PATH / "degradation_features_normalized.csv",
    index=False
)

# Save training-only normalization parameters
normalization_parameters = {
    "mean": descriptor_mean.to_dict(),
    "std": descriptor_std.to_dict()
}

with open(
    OUTPUT_PATH / "descriptor_normalization.json",
    "w"
) as file:
    json.dump(normalization_parameters, file, indent=4)

print("Normalized features saved successfully.")
print("Normalization parameters saved successfully.")


Normalized features saved successfully.
Normalization parameters saved successfully.


In [14]:

# STEP 9: FINAL FEATURE DATASET VERIFICATION

normalized_file = OUTPUT_PATH / "degradation_features_normalized.csv"
parameters_file = OUTPUT_PATH / "descriptor_normalization.json"

print("Normalized feature file exists:", normalized_file.exists())
print("Normalization parameter file exists:", parameters_file.exists())

print("\nSplit distribution:")
print(features_normalized["split"].value_counts())

print("\nClass distribution:")
print(
    features_normalized.groupby(["split", "class"]).size()
)

print("\nFinal feature table shape:")
print(features_normalized.shape)


Normalized feature file exists: True
Normalization parameter file exists: True

Split distribution:
split
train         7000
test          1500
validation    1500
Name: count, dtype: int64

Class distribution:
split       class
test        FAKE      750
            REAL      750
train       FAKE     3500
            REAL     3500
validation  FAKE      750
            REAL      750
dtype: int64

Final feature table shape:
(10000, 13)
